In [25]:
import cobra

In [26]:
# Load the model
model = cobra.io.read_sbml_model("../../model.xml")

In [27]:
# Define the medium without any carbon source
minimal_med = {
    "EX_cpd00007_e0": 20,  # O2_e0
    "EX_cpd00013_e0": 1000,  # Ammonia
    "EX_cpd00058_e0": 1000,  # Cu2+_e0
    "EX_cpd00971_e0": 1000,  # Na+_e0
    "EX_cpd00063_e0": 1000,  # Ca2+_e0
    "EX_cpd00048_e0": 1000,  # Sulfate_e0
    "EX_cpd10516_e0": 1000,  # fe3_e0
    "EX_cpd00254_e0": 1000,  # Mg_e0
    "EX_cpd00009_e0": 1000,  # Phosphate_e0
    "EX_cpd00205_e0": 1000,  # K+_e0
    "EX_cpd00013_e0": 1000,  # NH3_e0
    "EX_cpd00099_e0": 1000,  # Cl-_e0
    "EX_cpd00030_e0": 1000,  # Mn2+_e0
    "EX_cpd00001_e0": 1000,  # H2O_e0
    "EX_cpd00034_e0": 1000,  # Zn2+_e0
    "EX_cpd00149_e0": 1000,  # Co2+_e0
}

In [28]:
pro_exomets = [
    "cpd00023_e0",  # Glutamate (glutamic acid)
    "cpd00061_e0",  # PEP (phosphoenolpyruvate)
    "cpd00091_e0",  # UMP
]

In [34]:
exomet_rxns = {
    "cpd00023_e0": "EX_cpd00023_e0",  # Glutamate
    "cpd00061_c0": "SK_cpd00061_c0",  # PEP
    "cpd00091_c0": "SK_cpd00091_c0",  # UMP
}

In [29]:
# Check that the metabolites are in the model
for met_id in pro_exomets:
    try:
        model.metabolites.get_by_id(met_id)
    except KeyError:
        print(f"Metabolite {met_id} not found in the model.")

Metabolite cpd00061_e0 not found in the model.
Metabolite cpd00091_e0 not found in the model.


In [30]:
# Add sink reactions for the missing exometabolites
model.add_boundary(model.metabolites.cpd00061_c0, type="sink", lb=0)  # PEP
model.add_boundary(model.metabolites.cpd00091_c0, type="sink", lb=0)  # UMP

Reaction identifier,SK_cpd00091_c0
Name,UMP sink
Memory address,0x15db22950
Stoichiometry,cpd00091_c0 --> UMP -->
GPR,
Lower bound,0
Upper bound,1000.0


In [31]:
model.medium = minimal_med
model.reactions.EX_cpd00023_e0.lower_bound = -60/model.metabolites.cpd00023_c0.elements['C']
glut_sol = cobra.flux_analysis.pfba(model)

In [35]:
for met_id, rxn_id in exomet_rxns.items():
    model.medium = minimal_med
    # Set all the uptake rates to 0
    for rxn in exomet_rxns.values():
        model.reactions.get_by_id(rxn).lower_bound = 0
    # Correct the uptake rate for the current metabolite
    model.reactions.get_by_id(rxn_id).lower_bound = -60/model.metabolites.get_by_id(met_id).elements['C']
    sol = cobra.flux_analysis.pfba(model)
    print(f"Growth on {met_id}: {sol['bio1_biomass']}")

Growth on cpd00023_e0: 0.30495023998845067
Growth on cpd00061_c0: 0.5649126806358842
Growth on cpd00091_c0: 0.4741324908495703
